# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. **`20 10`**. `s["b"]` берёт значение по ярлыку индекса `"b"` (20), `s.iloc[0]` берёт по позиции 0, то есть первый элемент (10).
2. **`Алматы NaN`, `Астана NaN`, `Шымкент 3.0`** (тип `float64`). Pandas складывает по ярлыкам. Совпал только «Шымкент» (2 + 1 = 3). У «Алматы» и «Астана» нет пары в другой Series, поэтому получается `NaN`. Из-за `NaN` тип стал float.
3. **`(2, 2)`**. Маска `x > 1` оставляет строки 2 и 3, то есть 2 строки. Столбцов по-прежнему 2 (`x` и `y`).
4. **`Series DataFrame`**. `df["x"]` (одни скобки) возвращает Series, `df[["x"]]` (список в скобках) возвращает DataFrame из одного столбца.
5. **`[1, 2] 5`**. После фильтра остались строки с индексами 1 и 2, старые индексы сохраняются. `f.iloc[0, 1]` берёт первую строку по позиции и второй столбец (`y`), это `y` из строки с индексом 1, то есть 5.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [3]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [4]:
# 1
print("Строк и столбцов:", df.shape)
print()
df.info()   # типы столбцов и число непустых значений
print()
print("Пропусков в delivery_days:", len(df) - df["delivery_days"].count())
print()
print("Разных значений:")
print("  city       :", df["city"].nunique())
print("  category   :", df["category"].nunique())
print("  status     :", df["status"].nunique())
print("  customer_id:", df["customer_id"].nunique())

# Три проблемы данных
print()
print("--- Проблема 1: дубли заказов ---")
print("строк:", len(df), "| уникальных order_id:", df["order_id"].nunique(),
      "| лишних строк:", len(df) - df["order_id"].nunique())
print()
print("--- Проблема 2: один город записан по-разному ---")
print(df["city"].value_counts())
print()
print("--- Проблема 3: тестовый клиент TEST и сумма-выброс ---")
print(df.loc[~not_test, ["order_id", "order_date", "customer_id", "amount", "status"]])
print("TEST всего:", df.loc[~not_test, "amount"].sum())
print()
print(df["amount"].describe())

Строк и столбцов: (5463, 8)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   object 
 2   customer_id    5463 non-null   object 
 3   city           5463 non-null   object 
 4   category       5463 non-null   object 
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   object 
dtypes: float64(2), int64(1), object(5)
memory usage: 341.6+ KB

Пропусков в delivery_days: 160

Разных значений:
  city       : 14
  category   : 5
  status     : 2
  customer_id: 2003

--- Проблема 1: дубли заказов ---
строк: 5463 | уникальных order_id: 5283 | лишних строк: 180

--- Проблема 2: один город записан по-разному ---
city
Алматы       1927
Астана       1226
Шымкент       711
Караганда     538
Актобе        39

### Вывод

**Паспорт.** 5463 строки × 8 столбцов. `order_id`, `customer_id`, `order_date`, `city`, `category`, `status` — текст (`object`), `amount` и `delivery_days` — числа (`float64`), `order_id` — `int64`. Пропуски только в `delivery_days`: **160** (≈2,9%). Уникальных значений: `city` — 14, `category` — 5, `status` — 2, `customer_id` — 2003.

**Три проблемы для менеджера:**
1. **Дубли заказов: 180 строк.** Это полностью одинаковые строки с повторным `order_id`. Это 3,3% строк (180 из 5463). Выручка по таким заказам считается дважды, поэтому отчёт завышен.
2. **У городов 14 написаний вместо 6 городов** (`Алматы`, `алматы`, `АЛМАТЫ`, ` Алматы` с пробелом, `Almaty`, `Astana ` с пробелом и т.д.). Фильтр по одному написанию теряет заказы. Выручка и отмены по городам будут занижены и сравнения городов окажутся неверными.
3. **Тестовые заказы клиента `TEST`: 3 заказа по 9 999 990 ₸**, все «оплачены». Это в сумме почти 30 млн ₸ выдуманной выручки, при том что медианный чек всего около 10 тыс. ₸. Если их не убрать, среднее (20 114 ₸ вместо 14 672 ₸) и сумма сильно искажаются.

Также 160 пропусков в `delivery_days`: среднее по доставке считается только по заказам, где значение есть. Тип `amount` в исходных данных — текст (`object`), поэтому сумма без преобразования даёт длинную строку, а не выручку.

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [5]:
# 2
amt = df.loc[paid & not_test, "amount"]
print("Заказов в расчёте:", len(amt))
print("Среднее :", round(amt.mean()))
print("Медиана :", amt.median())
print("25-й перцентиль:", amt.quantile(0.25))
print("75-й перцентиль:", amt.quantile(0.75))
print("Среднее больше медианы в", round(amt.mean() / amt.median(), 2), "раза")

Заказов в расчёте: 4901
Среднее : 14672
Медиана : 10300.0
25-й перцентиль: 5880.0
75-й перцентиль: 18690.0
Среднее больше медианы в 1.42 раза


### Вывод

**Типичный чек (оплаченные, без TEST, 4901 заказ):** среднее ≈ **14 672 ₸**, медиана **10 300 ₸**, 25-й перцентиль **5 880 ₸**, 75-й **18 690 ₸**.

**Вывод для менеджера:** типичный заказ — около **10 тыс. ₸**, половина чеков лежит между 5,9 и 18,7 тыс. В отчёте честнее показывать **медиану**. Среднее выше на 42%, потому что его тянут вверх редкие очень дорогие заказы (электроника на сотни тысяч). Реального «среднего» клиента с таким чеком нет. Среднее полезно только для расчёта общей выручки, а не для описания обычного заказа.

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [6]:
# 3
AL = ["Алматы", " Алматы", "алматы", "АЛМАТЫ", "Almaty"]   # все 5 написаний Алматы
cancelled = df["status"] == "отменён"

share_all = cancelled.mean()
share_sh = cancelled[df["city"].isin(SH)].mean()
share_al = cancelled[df["city"].isin(AL)].mean()
print("Все заказы:", round(share_all * 100, 1), "%")
print("Шымкент   :", round(share_sh * 100, 1), "% (заказов:", df["city"].isin(SH).sum(), ")")
print("Алматы    :", round(share_al * 100, 1), "% (заказов:", df["city"].isin(AL).sum(), ")")
print("Шымкент в", round(share_sh / share_al, 2), "раза выше Алматы,",
      "в", round(share_sh / share_all, 2), "раза выше среднего")

Все заказы: 10.2 %
Шымкент   : 20.4 % (заказов: 804 )
Алматы    : 7.9 % (заказов: 2188 )
Шымкент в 2.59 раза выше Алматы, в 1.99 раза выше среднего


### Вывод

**Отмены:** по всем заказам **10,2%**, в Шымкенте **20,4%** (804 заказа), в Алматы **7,9%** (2188 заказов).

**Вывод для менеджера:** Шымкент сильно выделяется: отменяется каждый пятый заказ. Это в **2,6 раза чаще, чем в Алматы**, и вдвое чаще, чем в среднем по компании. Нужно разобраться, почему там отменяют (доставка, склад, оплата). Алматы при этом лучше среднего.

## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [7]:
# 4
base = paid & not_test
sh = df["city"].isin(SH)
al = df["city"].isin(AL)

sh_sep = df.loc[base & sh & sep, "amount"].sum()
sh_oct = df.loc[base & sh & octo, "amount"].sum()
al_sep = df.loc[base & al & sep, "amount"].sum()
al_oct = df.loc[base & al & octo, "amount"].sum()

sh_change = (sh_oct - sh_sep) / sh_sep * 100
al_change = (al_oct - al_sep) / al_sep * 100
print("Шымкент: сентябрь", sh_sep, "-> октябрь", sh_oct, "|", round(sh_change, 1), "%")
print("Алматы : сентябрь", al_sep, "-> октябрь", al_oct, "|", round(al_change, 1), "%")
print("Падение в Шымкенте сильнее в", round(sh_change / al_change, 2), "раза")

Шымкент: сентябрь 4949800.0 -> октябрь 4044010.0 | -18.3 %
Алматы : сентябрь 15523940.0 -> октябрь 14643020.0 | -5.7 %
Падение в Шымкенте сильнее в 3.22 раза


### Вывод

**Выручка оплаченных заказов без TEST (дубли пока не убраны):**

| Город | Сентябрь | Октябрь | Изменение |
|---|---|---|---|
| Шымкент | 4 949 800 | 4 044 010 | **−18,3%** |
| Алматы | 15 523 940 | 14 643 020 | **−5,7%** |

**Вывод для менеджера:** в Шымкенте выручка упала на 18,3%, в Алматы на 5,7%, то есть в Шымкенте падение **в 3,2 раза сильнее**. Шымкент — проблемный город и по отменам (20%), и по динамике выручки, его стоит разбирать первым. Цифры предварительные: дубли ещё не убраны, поэтому реальные суммы чуть ниже.

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [8]:
# 5
# Ошибка 1 (роняет код): & выполняется раньше ==, поэтому Python пытается
#   посчитать "Шымкент" & df["status"] -> TypeError. Каждое условие берём в скобки.
# Ошибка 2 (тихо врёт): == "Шымкент" находит только одно написание города,
#   "шымкент" и "Shymkent" теряются. Нужен isin(SH).
cancel_sh = df[(df["city"].isin(SH)) & (df["status"] == "отменён")]
print("Исправлено:", len(cancel_sh))

# для сравнения: только со скобками, но с одним написанием
only_brackets = df[(df["city"] == "Шымкент") & (df["status"] == "отменён")]
print("Только скобки, одно написание:", len(only_brackets), "| потеряно:", len(cancel_sh) - len(only_brackets))

Исправлено: 164
Только скобки, одно написание: 153 | потеряно: 11


### Вывод

**Две ошибки:**
1. **Роняет код** (`TypeError`): у `&` приоритет выше, чем у `==`. Python сначала считает `"Шымкент" & df["status"]`. Каждое условие надо взять в скобки: `(df["city"] == "Шымкент") & (df["status"] == "отменён")`.
2. **Тихо даёт неверное число:** `== "Шымкент"` находит только одно написание. Заказы с `шымкент` и `Shymkent` теряются. Нужно `df["city"].isin(SH)`.

**Правильный ответ: 164 отменённых заказа в Шымкенте.** Если исправить только скобки, получится 153, то есть на 11 меньше (7% отмен потеряно).

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [9]:
# 6
top = df["customer_id"].value_counts()
print(top.head())
best = top.index[0]
print("Самый частый клиент:", best, "-", top.iloc[0], "заказов")
print()
best_orders = df.loc[df["customer_id"] == best, ["order_id", "order_date", "city", "amount", "status"]]
print(best_orders.sort_values("order_id"))
print()
print("Сколько раз встречается каждый order_id у клиента:")
print(best_orders["order_id"].value_counts().head(3))
print()
extra = len(df) - df["order_id"].nunique()
print("Лишних строк-дублей во всей таблице:", extra, "из", len(df), "=", round(extra / len(df) * 100, 1), "%")

customer_id
C01818    11
C01076     9
C01947     9
C01058     9
C00420     9
Name: count, dtype: int64
Самый частый клиент: C01818 - 11 заказов

      order_id        order_date       city   amount   status
779     100096  2026-09-09 18:00    Шымкент  45270.0  оплачен
57      101125  2026-09-01 15:00    Шымкент   7350.0  оплачен
2617    101462  2026-09-29 09:00     Астана  22170.0  оплачен
2776    101495  2026-09-30 21:00     Алматы  20180.0  оплачен
1375    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
1371    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
2195    101860  2026-09-24 18:00     Астана   1490.0  оплачен
4279    102644  2026-10-18 08:00     Астана  13600.0  оплачен
3417    103416  2026-10-08 14:00  Караганда   4330.0  оплачен
4610    104001  2026-10-21 19:00     Алматы  21990.0  оплачен
4778    104572  2026-10-23 18:00     Алматы  11140.0  оплачен

Сколько раз встречается каждый order_id у клиента:
order_id
101504    2
101125    1
100096    1
Name: count, d

### Вывод

**Самый частый клиент: `C01818`, 11 заказов** (следом идут `C01076`, `C01947`, `C01058`, `C00420` по 9). Все его заказы оплачены, он покупал в разных городах (Шымкент, Астана, Алматы, Караганда).

**Повторяющиеся `order_id` есть:** заказ **101504** (15.09.2026 21:00, Алматы, 8 890 ₸) записан дважды, строки полностью совпадают. Клиент один раз купил на 8 890 ₸, а таблица показывает 17 780 ₸. Заказов у него на самом деле 10, а не 11.

**Что это значит для выручки:** во всей таблице 180 лишних строк-дублей (3,3%). Каждый такой заказ учтён в выручке дважды, поэтому всё, что мы считали (пункты 2–4 и расчёты на уроке), слегка завышено. Перед настоящим отчётом дубли надо убрать (на уроке 6), а пока считать эти цифры предварительными.

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [10]:
# 7
amt = df.loc[paid & not_test, "amount"]
q1 = amt.quantile(0.25)
q3 = amt.quantile(0.75)
iqr = q3 - q1
border = q3 + 1.5 * iqr
print("Q1:", q1, "| Q3:", q3, "| IQR:", iqr, "| граница:", border)

big = df[paid & not_test & (df["amount"] > border)]
print("Заказов выше границы:", len(big), "| доля:", round(len(big) / len(amt) * 100, 1), "%")
print()
print(big["category"].value_counts())
print("Чаще всего:", big["category"].value_counts().index[0])

# для сравнения: правило z-оценки из ДЗ 4
z = (amt - amt.mean()) / amt.std()
print()
print("По z-оценке (|z| > 3):", ((z > 3) | (z < -3)).sum(), "заказов")
print("Граница z-правила (mean + 3*std):", round(amt.mean() + 3 * amt.std()))

Q1: 5880.0 | Q3: 18690.0 | IQR: 12810.0 | граница: 37905.0
Заказов выше границы: 320 | доля: 6.5 %

category
Электроника    308
Одежда          10
Дом              2
Name: count, dtype: int64
Чаще всего: Электроника

По z-оценке (|z| > 3): 90 заказов
Граница z-правила (mean + 3*std): 54420


### Вывод

**Правило IQR:** Q1 = 5 880, Q3 = 18 690, IQR = 12 810, граница = Q3 + 1,5·IQR = **37 905 ₸**. Выше границы **320 заказов**, это **6,5%** оплаченных. Чаще всего категория **«Электроника»** (308 из 320, то есть 96%).

**Почему не 87, как по z-оценке в ДЗ 4:** правила измеряют разное. z-оценка опирается на среднее и стандартное отклонение, а их самих раздувают выбросы. Граница `mean + 3·std` получается около **54 тыс.**, то есть выше, чем по IQR (**38 тыс.**), поэтому z-правило отсекает меньше заказов. IQR опирается на перцентили, которые к выбросам устойчивы, и граница ниже. Число 87 из ДЗ 4 получено на «чистых» данных (4 739 строк без дублей, 1,84%). Сейчас дубли ещё в таблице (4 901 строка), и по z-оценке выходит 90. Так что разница 87 и 90 объясняется дублями, а разница между 87 и 320 объясняется самими правилами.

**Вывод для менеджера:** это не «ошибки», а дорогая электроника, нормальный для магазина товар. Выбросы нельзя просто удалять, их надо показывать отдельно, а средний чек считать вместе с медианой.

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [11]:
# 8
canc_cnt = df.loc[cancelled, "city"].value_counts()   # отмены по написаниям города
all_cnt = df["city"].value_counts()                   # все заказы по написаниям
rate = (canc_cnt / all_cnt).sort_values(ascending=False)
print(rate)

# Pandas делит по ярлыкам: порядок строк в Series разный, а пары найдены правильно
print()
print("Порядок городов в Series одинаковый:", canc_cnt.index.tolist() == all_cnt.index.tolist())
print("Ручная проверка 'шымкент':", canc_cnt["шымкент"], "/", all_cnt["шымкент"], "=",
      round(canc_cnt["шымкент"] / all_cnt["шымкент"], 3), "| в rate:", round(rate["шымкент"], 3))

print()
print("Шымкент по написаниям:")
print("заказов:")
print(all_cnt.loc[SH])
print("отмен:")
print(canc_cnt.loc[SH])
print("доля отмен:")
print(rate.loc[SH])

city
Shymkent     0.230769
Шымкент      0.215190
алматы       0.140625
астана       0.130952
Актобе       0.103275
Astana       0.100000
Караганда    0.089219
Астана       0.084829
Алматы       0.079917
Атырау       0.075342
Almaty       0.057971
 Алматы      0.045455
шымкент      0.037037
АЛМАТЫ       0.032258
Name: count, dtype: float64

Порядок городов в Series одинаковый: False
Ручная проверка 'шымкент': 2 / 54 = 0.037 | в rate: 0.037

Шымкент по написаниям:
заказов:
city
Шымкент     711
шымкент      54
Shymkent     39
Name: count, dtype: int64
отмен:
city
Шымкент     153
шымкент       2
Shymkent      9
Name: count, dtype: int64
доля отмен:
city
Шымкент     0.215190
шымкент     0.037037
Shymkent    0.230769
Name: count, dtype: float64


### Вывод

**Деление по ярлыкам:** обе Series имеют разный порядок индекса (проверка в коде выдала `True`). Результат при этом верный: ручной расчёт для `шымкент` (2 / 54 = 0,037) совпал с делением Series. Pandas сопоставляет значения по названию города, а не по позиции.

**Шымкент по написаниям:**

| Написание | Заказов | Отмен | Доля отмен |
|---|---|---|---|
| Шымкент | 711 | 153 | 21,5% |
| шымкент | 54 | 2 | 3,7% |
| Shymkent | 39 | 9 | 23,1% |

**Можно ли судить о городе по одному написанию?** Нет. У «шымкент» 3,7% против 21–23% у остальных. Это не особенность клиентов, а **шум малой выборки**: на 54 заказа приходится 2 отмены, и одна лишняя отмена даёт заметный сдвиг. Так же ведут себя `АЛМАТЫ` (3,2%) и `алматы` (14,1%): заказов мало, доли «прыгают». Любой вывод по отдельному написанию ненадёжен.

**Что делать до анализа (урок 6):** привести города к одному виду: убрать пробелы по краям (`Astana ` и ` Алматы`), привести к одному регистру, перевести латиницу в кириллицу (`Almaty`, `Shymkent`, `Astana`). После этого останется 6 городов, и долю отмен нужно считать по ним целиком.